In [1]:
import awkward as ak
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
from coffea.nanoevents.methods import vector

In [2]:
fileset = {
    "WJetsToLNu_Pt-250To400_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8____4_": 
    [
            "root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL18NanoAODv9/WJetsToLNu_Pt-250To400_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/130000/B349E45E-0BF8-8F43-895F-28D206346DB4.root",
#             "root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL18NanoAODv9/WJetsToLNu_Pt-250To400_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/130000/7FFCDC32-7714-6E4F-B67D-D5249140BD2F.root",
#             "root://cmsxrootd.fnal.gov//store/mc/RunIISummer20UL18NanoAODv9/WJetsToLNu_Pt-250To400_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8/NANOAODSIM/106X_upgrade2018_realistic_v16_L1v1-v1/130000/D62C3915-5A89-0245-B798-CA424003CE1E.root"
        ],
}
fname = fileset["WJetsToLNu_Pt-250To400_MatchEWPDG20_TuneCP5_13TeV-amcatnloFXFX-pythia8____4_"][0]
events = NanoEventsFactory.from_root(fname, schemaclass=NanoAODSchema).events()

In [4]:
E = events.Electron

In [5]:
E.pt

<Array [[], [133], ... [11.6, 6.59, 5.6]] type='1280444 * var * float32[paramete...'>

In [6]:
mask = E.pt>50

In [7]:
mask

<Array [[], [True], ... [False, False, False]] type='1280444 * var * bool'>

In [12]:
import awkward as ak
import numpy as np

In [13]:
ak.values_astype(mask, np.bool)

<Array [[], [True], ... [False, False, False]] type='1280444 * var * bool'>

In [14]:
len(E)

1280444

In [15]:
len(events)

1280444

In [17]:
ak.count(E, axis=-1)

<Array [{dEscaleDown: 0, ... photonIdxG: 3}] type='1280444 * {"dEscaleDown": int...'>

In [20]:
ak.num(E, axis=1)

<Array [0, 1, 1, 1, 0, 1, ... 2, 0, 1, 0, 2, 3] type='1280444 * int64'>

In [31]:
E[:,:1].pt

<Array [[], [133], [9.72, ... [229], [11.6]] type='1280444 * var * float32[param...'>

In [41]:
ak.argmax(E.pt, axis=-3)

ValueError: axis=-3 exceeds the depth of the nested list structure (which is 2)

(https://github.com/scikit-hep/awkward-1.0/blob/1.7.0/src/libawkward/Content.cpp#L1061)

In [42]:
Photon = events.Photon
phoMu, phoMuDR  = events.Photon.nearest(E,return_metric=True)
phoMuMask = ak.fill_none(phoMuDR > 0.4, True)

<Array [[], [0.0143, 2.84, ... [0.154, 0.0828]] type='1280444 * var * ?float32'>

In [44]:
j = events.Jet
j['p4'] = ak.zip({
"pt": j.pt,
"eta": j.eta,
"phi": j.phi,
"mass": j.mass},
with_name="PtEtaPhiMLorentzVector",
)

In [45]:
leading_j = j[:,:1]

In [67]:
leading_j.pt

<Array [[193], [233], ... [238], [201]] type='1280444 * var * float32[parameters...'>

In [57]:
E.pt[-1]

<Array [11.6, 6.59, 5.6] type='3 * float32[parameters={"__doc__": "p_{T}"}]'>

In [47]:
E['p4'] = ak.zip({
                    "pt": E.pt,
                    "eta": E.eta,
                    "phi": E.phi,
                    "mass": E.mass},
                    with_name="PtEtaPhiMLorentzVector",
)

In [61]:
E.metric_table(leading_j)

<Array [[], [[0.0154]], ... [0.267], [3.63]]] type='1280444 * var * var * float32'>

In [62]:
ljEPairs = ak.cartesian({"lj":leading_j, "ele": E})

In [82]:
ljEPairs.lj.delta_r(ljEPairs.ele) < 3.4

<Array [[], [True], ... [False, True, False]] type='1280444 * var * bool'>

In [87]:
ak.any(ljEPairs.lj.delta_phi(ljEPairs.ele) < 3.4, axis=-1)

<Array [False, True, True, ... True, True] type='1280444 * bool'>

In [68]:
ak.cartesian(leading_j, E)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/cvmfs/sft.cern.ch/lcg/views/LCG_96python3/x86_64-centos7-gcc8-opt/lib/python3.6/site-packages/IPython/core/interactiveshell.py", line 3296, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-68-631137f72bdd>", line 1, in <module>
    ak.cartesian(leading_j, E)
  File "/cms/ldap_home/runiyal/.local/lib/python3.6/site-packages/awkward/operations/structure.py", line 3182, in cartesian
    behavior = ak._util.behaviorof(*arrays, behavior=behavior)
  File "/cms/ldap_home/runiyal/.local/lib/python3.6/site-packages/awkward/highlevel.py", line 582, in __iter__
    yield ak._util.wrap(x, self._behavior)
  File "/cms/ldap_home/runiyal/.local/lib/python3.6/site-packages/awkward/_util.py", line 520, in wrap
    return ak.highlevel.Array(content, behavior=behavior, kernels=None)
  File "/cms/ldap_home/runiyal/.local/lib/python3.6/site-packages/awkward/highlevel.py", line 276, in __init__
    docstr = self.layout.purel


KeyboardInterrupt

